###Ingest races.csv file

#####Step 1 - Read the CSV file using the spark dataframe reader

In [0]:
 races_df = spark.read.option("header", True).csv("abfss://raw@dformulaone.dfs.core.windows.net/races.csv")

#####Step 2 - Set Schema

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

In [0]:
races_schema = StructType(fields=[StructField("raceid", IntegerType(), False),
                                    StructField("year", IntegerType(), True), 
                                    StructField("round", IntegerType(), True), 
                                    StructField("circuitid", IntegerType(), True),
                                    StructField("name", StringType(), True),
                                    StructField("date", TimestampType(), True), 
                                    StructField("time", TimestampType(), True), 
                                    StructField("url", StringType(), True)
])

In [0]:
 races_df = spark.read \
 .option("header", True) \
 .schema(races_schema) \
 .csv("abfss://raw@dformulaone.dfs.core.windows.net/races.csv")

#####Step 3 - Rename the columns as required 

In [0]:
races_renamed_df = races_df \
    .withColumnRenamed("raceid", "race_id") \
    .withColumnRenamed("year", "race_year") \
    .withColumnRenamed("circuitid", "circuit_id")

#####Step 4 -  Add race_timestamp and ingestion_date to the dataframe

In [0]:
from pyspark.sql.functions import current_timestamp , lit, to_timestamp, concat, col

In [0]:
races_transformed_df = races_renamed_df \
    .withColumn("race_timestamp", to_timestamp(concat(col("date"), lit(" "), col("time")), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("ingestion_date", current_timestamp())

In [0]:
races_selected_df = races_transformed_df.select(col("race_id"), col("race_year"), col("round"), col("circuit_id"), col("name"), col("race_timestamp"), col("ingestion_date"))

#####Step 5 - Write data to datalake as parquet

In [0]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
races_selected_df.write.mode("overwrite").parquet("abfss://processed@dformulaone.dfs.core.windows.net/races")

In [0]:
%fs
ls abfss://processed@dformulaone.dfs.core.windows.net/races

In [0]:
# df = spark.read.parquet("abfss://processed@dformulaone.dfs.core.windows.net/races")
# display(df, truncate=False)

#####Step 5.1 - Partition by race_year

In [0]:
races_selected_df.write.mode("overwrite").partitionBy("race_year").parquet("abfss://processed@dformulaone.dfs.core.windows.net/races")

In [0]:
%fs
ls abfss://processed@dformulaone.dfs.core.windows.net/races